# Sum from 0 through n: a timing testbench

Instructor notebook for comparing a closed form, a loop, and recursion. It starts with small correctness checks, then measures how runtime changes as `n` grows. The benchmark targets about 35 seconds and has a 95-second safety stop, leaving time to draw the graphs within the requested two-minute window on a normal run. Machine load and Python speed affect the actual duration.

The goal is to compare evidence with a growth model, not to declare a universal winner from one stopwatch reading. Run the cells in order.

## The three implementations

The recursive function has a base case at zero. Each call reduces `n` by one, so it moves toward that case. We will use a plain loop to display results clearly instead of compressing the comparison into a one-line expression.

In [ ]:
def sum_formula(n):
    if n < 0:
        raise ValueError("n must be nonnegative")
    return n * (n + 1) // 2


def sum_loop(n):
    if n < 0:
        raise ValueError("n must be nonnegative")
    total = 0
    for value in range(n + 1):
        total += value
    return total


def sum_recursive(n):
    if n < 0:
        raise ValueError("n must be nonnegative")
    if n == 0:                 # base case
        return 0
    return n + sum_recursive(n - 1)  # smaller subproblem


## First, check the answer

For each small input, show the three answers side by side. This is an easy way to catch a missing endpoint or an incorrect base case before measuring speed.

In [ ]:
small_case_results = []

for n in range(8):
    formula_answer = sum_formula(n)
    loop_answer = sum_loop(n)
    recursive_answer = sum_recursive(n)

    if formula_answer != loop_answer or loop_answer != recursive_answer:
        raise AssertionError(f"The methods disagree when n = {n}")

    row = {
        "n": n,
        "closed form": formula_answer,
        "loop": loop_answer,
        "recursive": recursive_answer,
    }
    small_case_results.append(row)

for row in small_case_results:
    print(row)

## Derive a growth bound for each function

We use the unit-cost model first: one scalar arithmetic operation, comparison, or function call costs one abstract step. Big O is an asymptotic upper bound; when matching upper and lower bounds give the same order, we can say the tighter Theta bound as well. These derivations tell us what curve to compare with measurements.

### Closed form: O(1) arithmetic operations

The return expression is n * (n + 1) // 2. Count one addition, one multiplication, and one integer division: a fixed total of 3 arithmetic operations, regardless of n. So T_formula(n) = 3 in this model, which is O(1) (and Theta(1)). The input check is also constant work. This model treats integers as fixed-size values; Python integers can have many digits, so the real cost of arithmetic slowly rises with n's bit length.

### Loop: O(n) additions

The loop visits 0, 1, ..., n: that is n + 1 iterations. If each body addition costs c, then T_loop(n) = c(n + 1) plus loop-control work. For n >= 1, c(n + 1) is bounded above and below by constant multiples of n, so T_loop(n) is Theta(n), and therefore O(n). The sequence of addition counts is 1, 2, 3, 4, ... as n is 0, 1, 2, 3, ... . Extra space is O(1).

### Recursion: O(n) calls and additions

Let C(n) count calls, including the base-case call. C(0) = 1, and for n > 0, C(n) = C(n - 1) + 1. Expanding gives C(n) = 1 + n = n + 1. Let A(n) count additions: A(0) = 0 and A(n) = A(n - 1) + 1, so A(n) = n. If each call has constant overhead a and each addition costs b, then T_recursive(n) = a(n + 1) + bn, which is Theta(n), hence O(n). The active call stack also reaches depth n + 1, so extra space is O(n).

### What the derivation predicts

The loop and recursive function both have linear growth, but Big O does not say their seconds will match: recursive calls carry larger constant overhead. The formula's abstract operation count stays flat, though arbitrary-precision integer arithmetic means its measured time need not be perfectly flat. The count tables and graphs below let us compare the declared model, exact operation counts, and observed runtime—and see where each description is useful.

## Derive the count, then count the work

Big O comes from a cost model. For this comparison, count scalar additions, loop-body iterations, and recursive function calls; do not count the bookkeeping used to update the counters. The closed form has three arithmetic operators in its expression (`n + 1`, multiplication, integer division), so this model counts 3. The loop has n + 1 iterations and additions. The recursive function makes n + 1 calls, including the base case, and performs n additions.

These counts form sequences. For recursive calls, C(0) = 1 and C(n) = C(n - 1) + 1, so C(n) = n + 1. For recursive additions, A(0) = 0 and A(n) = A(n - 1) + 1, so A(n) = n. Both are Theta(n). This explicit model explains what the asymptotic claim counts—and what it leaves out, such as Python call overhead and the growing bit length of integers.

In [ ]:
def sum_loop_counted(n):
    total = 0
    loop_iterations = 0
    additions = 0
    for value in range(n + 1):
        loop_iterations += 1
        total += value
        additions += 1
    return total, loop_iterations, additions


def sum_recursive_counted(n):
    call_count = 1
    addition_count = 0

    def visit(k):
        nonlocal call_count, addition_count
        if k == 0:
            return 0
        call_count += 1
        addition_count += 1
        return k + visit(k - 1)

    result = visit(n)
    return result, call_count, addition_count


count_sizes = (0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512)
operation_count_rows = []

for n in count_sizes:
    loop_answer, loop_iterations, loop_additions = sum_loop_counted(n)
    recursive_answer, recursive_calls, recursive_additions = sum_recursive_counted(n)
    if loop_answer != sum_formula(n) or recursive_answer != sum_formula(n):
        raise AssertionError(f"Counted implementation disagrees when n = {n}")

    row = {
        "n": n,
        "formula arithmetic ops": 3,
        "loop iterations": loop_iterations,
        "loop additions": loop_additions,
        "recursive calls": recursive_calls,
        "recursive additions": recursive_additions,
    }
    operation_count_rows.append(row)

for row in operation_count_rows:
    print(row)

The counters now agree with the derivation: loop work grows with n, recursive call count grows with n, and the formula's abstract operation count is flat. The counter variables add bookkeeping instructions, so do not use these instrumented functions for the timing graph. They answer a different question: how many chosen operations does the algorithm perform? The original functions answer how long this implementation took on this computer.

## Benchmark plan

We will time all three methods on modest inputs, where recursion is safe, and compare the formula with the loop on inputs up to six million. The recursive function is intentionally not pushed toward Python's call-depth limit; reaching that limit would measure a language constraint rather than useful runtime growth.

To make tiny formula calls measurable, the testbench times a batch and divides by the number of calls. The loop and recursion are timed as individual calls. Each sweep records another observation, and the final graphs use medians to reduce the influence of a brief background interruption. The run continues for at least 35 seconds and stops at 95 seconds if the machine is unusually slow.

In [ ]:
from math import ceil
from statistics import median
from time import perf_counter

small_sizes = (25, 50, 100, 200, 400, 700)
large_sizes = (10_000, 50_000, 200_000, 800_000, 2_000_000, 6_000_000)
formula_batch_size = 2_000

measurements = {
    "formula_small": {n: [] for n in small_sizes},
    "loop_small": {n: [] for n in small_sizes},
    "recursive_small": {n: [] for n in small_sizes},
    "formula_large": {n: [] for n in large_sizes},
    "loop_large": {n: [] for n in large_sizes},
}

# Calibrate a few loop repetitions so each sweep does visible work.
calibration_start = perf_counter()
sum_loop(large_sizes[-1])
calibration_seconds = perf_counter() - calibration_start
loop_repetitions = max(1, min(4, ceil(0.06 / calibration_seconds)))

target_seconds = 35.0
safety_stop_seconds = 95.0
benchmark_start = perf_counter()
sweep_count = 0

while True:
    elapsed_before_sweep = perf_counter() - benchmark_start
    if elapsed_before_sweep >= safety_stop_seconds:
        break
    if elapsed_before_sweep >= target_seconds:
        break

    for n in small_sizes:
        formula_start = perf_counter()
        for _ in range(formula_batch_size):
            sum_formula(n)
        formula_elapsed = perf_counter() - formula_start
        measurements["formula_small"][n].append(formula_elapsed / formula_batch_size)

        loop_start = perf_counter()
        sum_loop(n)
        measurements["loop_small"][n].append(perf_counter() - loop_start)

        recursive_start = perf_counter()
        sum_recursive(n)
        measurements["recursive_small"][n].append(perf_counter() - recursive_start)

    for n in large_sizes:
        formula_start = perf_counter()
        for _ in range(formula_batch_size):
            sum_formula(n)
        formula_elapsed = perf_counter() - formula_start
        measurements["formula_large"][n].append(formula_elapsed / formula_batch_size)

        loop_start = perf_counter()
        for _ in range(loop_repetitions):
            sum_loop(n)
        loop_elapsed = perf_counter() - loop_start
        measurements["loop_large"][n].append(loop_elapsed / loop_repetitions)

    sweep_count += 1

benchmark_seconds = perf_counter() - benchmark_start
print(f"Completed {sweep_count} measurement sweeps in {benchmark_seconds:.1f} seconds.")
print(f"Large-loop repetitions per input: {loop_repetitions}")
print(f"Observations per input: {len(measurements['loop_large'][large_sizes[0]])}")

## Summarize the recorded times

Each entry below is the median of the repeated measurements for that input size. Values are seconds per function call. The formula uses batched timing, so dividing by the batch size reduces timer-resolution noise.

In [ ]:
timing_table = []

for n in small_sizes:
    row = {
        "n": n,
        "formula seconds": median(measurements["formula_small"][n]),
        "loop seconds": median(measurements["loop_small"][n]),
        "recursive seconds": median(measurements["recursive_small"][n]),
    }
    timing_table.append(row)

for n in large_sizes:
    row = {
        "n": n,
        "formula seconds": median(measurements["formula_large"][n]),
        "loop seconds": median(measurements["loop_large"][n]),
        "recursive seconds": None,
    }
    timing_table.append(row)

for row in timing_table:
    print(
        f"n={row['n']:>9,} | "
        f"formula={row['formula seconds']:.9f}s | "
        f"loop={row['loop seconds']:.6f}s | "
        f"recursive={row['recursive seconds'] if row['recursive seconds'] is not None else 'depth-limited'}"
    )

## Graph actual time and the Big O model

The first graph plots recorded seconds against n and overlays reference curves scaled to the first large-input measurement: flat for O(1), and proportional to n for O(n). The second compares observed relative growth with those same shapes on the range where all three methods can be measured. This scaling lets us compare actual runtime with predicted shape without pretending Big O predicts the actual seconds.

The recursive series stops at n = 700 because Python's call stack is finite. That endpoint is useful evidence about the implementation environment, but it does not change the O(n) classification before the limit.

In [ ]:
import math
from IPython.display import SVG, display

small_formula_times = [median(measurements["formula_small"][n]) for n in small_sizes]
small_loop_times = [median(measurements["loop_small"][n]) for n in small_sizes]
small_recursive_times = [median(measurements["recursive_small"][n]) for n in small_sizes]
large_formula_times = [median(measurements["formula_large"][n]) for n in large_sizes]
large_loop_times = [median(measurements["loop_large"][n]) for n in large_sizes]

def make_log_chart(title, x_label, y_label, series, x_bounds=None, y_bounds=None):
    width, height = 860, 540
    left, right, top, bottom = 86, 28, 52, 130
    plot_width = width - left - right
    plot_height = height - top - bottom
    all_x = [x for item in series for x in item["x"]]
    all_y = [y for item in series for y in item["y"] if y > 0]
    low_x, high_x = x_bounds or (min(all_x), max(all_x))
    low_y, high_y = y_bounds or (min(all_y), max(all_y))
    log_low_x, log_high_x = math.log10(low_x), math.log10(high_x)
    log_low_y, log_high_y = math.log10(low_y), math.log10(high_y)

    def px(value):
        return left + (math.log10(value) - log_low_x) / (log_high_x - log_low_x) * plot_width

    def py(value):
        return top + plot_height - (math.log10(value) - log_low_y) / (log_high_y - log_low_y) * plot_height

    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" role="img" aria-label="{title}">']
    parts.append('<rect width="100%" height="100%" rx="16" fill="#fffefa"/>')
    parts.append(f'<text x="{left}" y="30" font-size="19" font-weight="700" fill="#172a35">{title}</text>')

    for tick_index in range(5):
        fraction = tick_index / 4
        x_value = 10 ** (log_low_x + fraction * (log_high_x - log_low_x))
        x_pos = left + fraction * plot_width
        parts.append(f'<line x1="{x_pos:.1f}" y1="{top}" x2="{x_pos:.1f}" y2="{top + plot_height}" stroke="#e1e7e2"/>')
        parts.append(f'<text x="{x_pos:.1f}" y="{top + plot_height + 20}" text-anchor="middle" font-size="11" fill="#536772">{x_value:,.0f}</text>')
        y_value = 10 ** (log_low_y + fraction * (log_high_y - log_low_y))
        y_pos = top + plot_height - fraction * plot_height
        parts.append(f'<line x1="{left}" y1="{y_pos:.1f}" x2="{left + plot_width}" y2="{y_pos:.1f}" stroke="#e1e7e2"/>')
        parts.append(f'<text x="{left - 10}" y="{y_pos + 4:.1f}" text-anchor="end" font-size="11" fill="#536772">{y_value:.1g}</text>')

    parts.append(f'<text x="{left + plot_width / 2}" y="{height - 15}" text-anchor="middle" font-size="13" fill="#172a35">{x_label} · log scale</text>')
    parts.append(f'<text transform="translate(20 {top + plot_height / 2}) rotate(-90)" text-anchor="middle" font-size="13" fill="#172a35">{y_label} · log scale</text>')

    legend_x, legend_y = left, top + plot_height + 32
    legend_row = 0
    for item in series:
        points = " ".join(f"{px(x):.1f},{py(y):.1f}" for x, y in zip(item["x"], item["y"]) if y > 0)
        color = item["color"]
        dash = item.get("dash", "none")
        parts.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2.5" stroke-dasharray="{dash}"/>')
        for x, y in zip(item["x"], item["y"]):
            if y > 0:
                parts.append(f'<circle cx="{px(x):.1f}" cy="{py(y):.1f}" r="3.5" fill="{color}"/>')
        label = item["label"]
        legend_width = 22 + len(label) * 6.4 + 20
        if legend_x + legend_width > width - right:
            legend_x = left
            legend_row += 1
            legend_y += 20
        parts.append(f'<line x1="{legend_x}" y1="{legend_y}" x2="{legend_x + 18}" y2="{legend_y}" stroke="{color}" stroke-width="3" stroke-dasharray="{dash}"/>')
        parts.append(f'<text x="{legend_x + 23}" y="{legend_y + 4}" font-size="11" fill="#172a35">{label}</text>')
        legend_x += legend_width

    parts.append('</svg>')
    return SVG("".join(parts))

large_formula_reference = [large_formula_times[0] for _ in large_sizes]
large_loop_reference = [large_loop_times[0] * n / large_sizes[0] for n in large_sizes]

actual_series = [
    {"label": "Formula · small", "x": list(small_sizes), "y": small_formula_times, "color": "#087e78"},
    {"label": "Loop · small", "x": list(small_sizes), "y": small_loop_times, "color": "#3567a6"},
    {"label": "Recursive", "x": list(small_sizes), "y": small_recursive_times, "color": "#c94f36"},
    {"label": "Formula · large", "x": list(large_sizes), "y": large_formula_times, "color": "#087e78", "dash": "6 4"},
    {"label": "Loop · large", "x": list(large_sizes), "y": large_loop_times, "color": "#3567a6", "dash": "6 4"},
    {"label": "Formula O(1) reference", "x": list(large_sizes), "y": large_formula_reference, "color": "#087e78", "dash": "2 4"},
    {"label": "Loop O(n) reference", "x": list(large_sizes), "y": large_loop_reference, "color": "#3567a6", "dash": "2 4"},
]
display(make_log_chart("Measured runtime vs. Big O-shaped references", "Input size n", "Seconds per call", actual_series))

def relative(values):
    baseline = values[0]
    return [value / baseline for value in values]

n_scale = [n / small_sizes[0] for n in small_sizes]
growth_series = [
    {"label": "Formula · measured", "x": list(small_sizes), "y": relative(small_formula_times), "color": "#087e78"},
    {"label": "O(1) reference", "x": list(small_sizes), "y": [1.0 for _ in small_sizes], "color": "#172a35", "dash": "6 4"},
    {"label": "Loop · measured", "x": list(small_sizes), "y": relative(small_loop_times), "color": "#3567a6"},
    {"label": "O(n) reference", "x": list(small_sizes), "y": n_scale, "color": "#536772", "dash": "2 4"},
    {"label": "Recursive · measured", "x": list(small_sizes), "y": relative(small_recursive_times), "color": "#c94f36"},
]
display(make_log_chart("Observed growth vs. Big O shape", "Input size n", "Relative time", growth_series, x_bounds=(small_sizes[0], small_sizes[-1])))

positive_count_sizes = list(count_sizes[1:])
formula_operation_counts = [row["formula arithmetic ops"] for row in operation_count_rows[1:]]
loop_addition_counts = [row["loop additions"] for row in operation_count_rows[1:]]
recursive_addition_counts = [row["recursive additions"] for row in operation_count_rows[1:]]
recursive_call_counts = [row["recursive calls"] for row in operation_count_rows[1:]]
count_series = [
    {"label": "Formula ops · counted", "x": positive_count_sizes, "y": formula_operation_counts, "color": "#087e78"},
    {"label": "O(1) reference", "x": positive_count_sizes, "y": [3 for _ in positive_count_sizes], "color": "#172a35", "dash": "6 4"},
    {"label": "Loop additions", "x": positive_count_sizes, "y": loop_addition_counts, "color": "#3567a6"},
    {"label": "Recursive additions", "x": positive_count_sizes, "y": recursive_addition_counts, "color": "#c94f36"},
    {"label": "Recursive calls", "x": positive_count_sizes, "y": recursive_call_counts, "color": "#dda62e"},
    {"label": "O(n) reference", "x": positive_count_sizes, "y": positive_count_sizes, "color": "#536772", "dash": "2 4"},
]
display(make_log_chart("Counted operations and asymptotic references", "Input size n", "Operation count", count_series, x_bounds=(positive_count_sizes[0], positive_count_sizes[-1])))

## Does Big O help here?

Big O helps us anticipate the broad trend. Doubling n should roughly double the work for the loop and recursive method, while the closed form should stay nearly flat under the introductory arithmetic-operation model. This tells us which approach is likely to scale better before we run it.

The measurements add details Big O leaves out: recursion has more overhead than the loop, very small timings are noisy, hardware changes seconds, and Python's integer arithmetic grows with the number of digits. Big O does not tell us the exact runtime, the crossover point, or whether a particular run is affected by other work on the computer. The plot is evidence about these implementations on this machine; the growth class is the model we use to reason beyond the measured points.

### Discuss

- Do the loop and recursive measurements have similar broad growth? Why might one be consistently slower?
- Does the formula line look perfectly flat? What work is Big O setting aside?
- If the recursive version is O(n), why can it fail at an n that the loop handles easily?
- Which claims come from the asymptotic model, and which claims come only from this experiment?